# PyTorch XLM-RoBERTa base

I will fine-tune `FacebookAI/xlm-roberta-base` with its matching tokenizer for three-way natural-language inference.

Reusable loading, splitting, batching, training, and evaluation code lives in `scripts/watson_model_helpers.py`.

The premise-grouped split evaluates generalization to unseen premises.

## Venv and requirements

Different Python versions were used during development depending on hardware and software constraints. Please see the repository’s [setup documentation](../README.md#setup) for details.

## Model and tokenizer choice

- starting from general pretrained [XLM-RoBERTa base checkpoint](https://huggingface.co/FacebookAI/xlm-roberta-base)
- newly initialized three-label classification head. 
- The tokenizer receives premise first, hypothesis second, with dynamic batch padding and configurable truncation.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "watson_dataset_config.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from watson_dataset_helpers import load_datasets
from watson_model_helpers import (
    split_training_data,
    load_model_and_tokenizer,
    tokenize_data,
    make_loaders,
    fine_tune,
    check_bf16_support
)

## Bring in the data

In [2]:
datasets = load_datasets()

training_data = datasets["train"].copy()
test_data = datasets["test"].copy()

Kaggle files already present; using: (project_root)/data/raw


## Training-data preparation and validation split

### Goals in Preparing Training/Validation split:
- No exact premise appears in both training and validation. All rows sharing a premise stay together.
- Have roughly the same label proportions in validation and the training side as a whole.

Note: Proportions are approximate because premise groups stay intact.

In [3]:
training, validation = split_training_data(training_data)

## Data Cleaning

Prior exploration in `01_dataset_exploration.ipynb` found

- no missing values
- no duplicate premise–hypothesis pairs

No further text cleaning is planned; the original premise and hypothesis text will be kept.

## Bring in Pre-Trained Model

### XLM-Roberta Base
- XLM-Roberta Base is chosen as the pre-trained model.
- XLM-Roberta seems like a natural first check.
- Base model chosen to baseline performance using, and acknowledging, local HW constraints
- Other models may be used in the future, either in this notebook or separate notebooks

In [4]:
# Run before model initialization and loader creation for each fresh experiment.
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)  # Seeds CPU and accelerator RNGs, including MPS.
# A fixed seed does not guarantee fully deterministic MPS execution.


In [5]:
model, tokenizer = load_model_and_tokenizer()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Tokenization

In [6]:
training_encodings = tokenize_data(training, tokenizer)
validation_encodings = tokenize_data(validation, tokenizer)

## Batching, Padding, Training Settings

In [7]:
batch_size = 4  # sweet spot on my HW
# learning_rate = 2e-5
# learning_rate = 1e-5
#learning_rate = 1.5e-5
#learning_rate = 5e-6
#learning_rate = 2.5e-6
learning_rate = 7.5e-6 
# learning_rate = 6.25e-6
# learning_rate = 6.875e-6
epochs = 5
weight_decay = 0.05  # Previous AdamW default: 0.01.
use_bf16 = False  # Training precision.
validation_use_bf16 = False  # Independent validation precision; True enables BF16.

# Each training call creates a unique run folder here; best.pt persists on disk.
checkpoint_dir = PROJECT_ROOT / "data" / "checkpoints" / "xlm-roberta-base"
early_stopping_patience = 2  # Consecutive non-improving epochs; None disables stopping.
min_delta = 0.0  # Minimum loss decrease that resets patience.
# The best weights are restored automatically; checkpoints are not resumable runs.

training_loader, validation_loader = make_loaders(
    training_encodings, validation_encodings, tokenizer, batch_size
)

### GPU memory investigation

Apple MPS out-of-memory issues were encountered during development. Please visit [MPS memory investigation](../docs/investigations/mps-memory.md) for the issues encountered and their resolutions. Please also visit [runtime performance documentation](../docs/runtimes/README.md) for performance findings from the MPS memory investigation.

## Fine-tuning

In [8]:
if use_bf16 or validation_use_bf16:
    check_bf16_support()

training_result = fine_tune(
    model, training_loader, validation_loader, learning_rate, epochs,
    use_bf16=use_bf16, validation_use_bf16=validation_use_bf16,
    checkpoint_dir=checkpoint_dir, weight_decay=weight_decay,
    early_stopping_patience=early_stopping_patience, min_delta=min_delta,
)


2026-09-21 20:00:42 | Training on mps, 2424 batches/epoch, 5 epochs, learning rate 7.5e-06, weight decay 0.05, training precision FP32; validation precision FP32
Best checkpoint: /Users/kurtmeehanpro/Documents/code/git/lying-detective/data/checkpoints/xlm-roberta-base/run-20260921-200042-npdt8xzj/best.pt
Early stopping: patience=2, min_delta=0.0
2026-09-21 20:00:42 | Processing batch 0/2424
2026-09-21 20:02:39 | Processing batch 500/2424
2026-09-21 20:04:35 | Processing batch 1000/2424
2026-09-21 20:06:34 | Processing batch 1500/2424
2026-09-21 20:08:30 | Processing batch 2000/2424
2026-09-21 20:10:23 | Epoch 1/5: training loss = 1.0915, validation loss = 1.0191, validation accuracy = 47.20%
Saved best checkpoint: epoch 1, validation loss=1.0191
2026-09-21 20:10:24 | Processing batch 0/2424
2026-09-21 20:12:20 | Processing batch 500/2424
2026-09-21 20:14:17 | Processing batch 1000/2424
2026-09-21 20:16:14 | Processing batch 1500/2424
2026-09-21 20:18:11 | Processing batch 2000/2424
202

## Evaluation

## Next observations
